# Stage 2.3a - Four-arm decoder capacity and save/resume gate

This notebook is the authoritative pre-cross-validation capacity gate. It consumes the exact
fold-0 frozen-feature cache and runs the two approved subsets across the complete 2x2 factorial:
ReLU/plain, PReLU/residual, ReLU/residual, and PReLU/plain. The decoder is byte-identical in
architecture to the final comparison decoder: four-channel logits at 128^3 followed by one shared
trilinear upsample to the 256^3 target grid.

## Entry and success criteria

1. Fold 0 has an independent decoder-entry QA PASS bound to the current P2 audit and protocol
   amendment hashes.
2. All eight arm/subset runs reach hard Dice >= 0.90 for every case and bone at two consecutive
   evaluations.
3. Save/reload preserves parameters and pre-update logits and training continues.
4. All runs use the identical frozen-feature-cache SHA-256 and retain at least 10% GPU headroom.
5. `foundation_summary.json` reports exactly eight successful results before combined Stage 2
   review may unlock full cross-validation.


In [ ]:
import contextlib
import hashlib
import json
import os
import platform
import random
try:
    import resource
except ImportError:
    resource = None
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

STAGE2_SCHEMA = "foundation_stage2_v1"
PROTOCOL_VERSION = "baseline_protocol_v1"
SEED = 42
FOLD = 0
RUN_DECODER_GATE = False
TARGET_SIZE = 256
MAX_EPOCHS = 500
EVALUATE_EVERY = 10
REQUIRED_CONSECUTIVE_PASSES = 2
MINIMUM_DICE_EACH_CASE_BONE = 0.90
LEARNING_RATE = 1e-3
USE_AMP = True
USE_ACTIVATION_CHECKPOINTING = True
BONES = ["femur", "tibia", "patella", "fibula"]
FEATURE_CHANNELS = [64, 128, 256, 512]
DECODER_LOGIT_RESOLUTION = 128
ARM_FACTORS = {
    "plain_unet_style":    {"activation": "relu",  "residual": False, "label": "U (ReLU, plain)"},
    "residual_vnet_style": {"activation": "prelu", "residual": True,  "label": "V (PReLU, residual)"},
    "residual_relu_style": {"activation": "relu",  "residual": True,  "label": "ReLU + residual"},
    "plain_prelu_style":   {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
}
ARMS = list(ARM_FACTORS)
SUBSETS = {"subset_1_healthy": ["VSD_016_Left"], "subset_2_mixed": ["VSD_016_Left", "Case14_PartRight"]}

assert FOLD == 0 and TARGET_SIZE == 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(DEVICE), "run_decoder_gate": RUN_DECODER_GATE})

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt, label

BONES = ["femur", "tibia", "patella", "fibula"]


def overlap_metrics(prediction, target):
    """Explicit empty handling: an empty target is invalid; an empty prediction against a target scores zero."""
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    invalid_target = not target.any(); empty_prediction = not prediction.any()
    if invalid_target:
        return {"dice": float("nan"), "iou": float("nan"), "invalid_target": True, "empty_prediction": empty_prediction}
    if empty_prediction:
        return {"dice": 0.0, "iou": 0.0, "invalid_target": False, "empty_prediction": True}
    intersection = np.logical_and(prediction, target).sum(dtype=np.float64)
    pred_count = prediction.sum(dtype=np.float64); target_count = target.sum(dtype=np.float64)
    return {"dice": float(2 * intersection / (pred_count + target_count)), "iou": float(intersection / (pred_count + target_count - intersection)), "invalid_target": False, "empty_prediction": False}


def _surface_distances(prediction, target, spacing_xyz):
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    spacing_xyz = tuple(float(value) for value in spacing_xyz)
    if len(spacing_xyz) != 3 or any(value <= 0 for value in spacing_xyz): raise ValueError("spacing_xyz must contain three positive millimetre values")
    pred_surface = prediction & ~binary_erosion(prediction); target_surface = target & ~binary_erosion(target)
    if not pred_surface.any() or not target_surface.any(): return None
    to_target = distance_transform_edt(~target_surface, sampling=spacing_xyz)[pred_surface]
    to_prediction = distance_transform_edt(~pred_surface, sampling=spacing_xyz)[target_surface]
    return np.concatenate([to_target, to_prediction])


def hd95_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(np.percentile(distances, 95))


def assd_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(distances.mean())


def per_bone_metrics(logits, target, spacing_xyz=(0.78125, 0.78125, 0.78125), threshold=0.5):
    probability = torch.sigmoid(logits.float()).detach().cpu().numpy(); truth = target.detach().cpu().numpy() > 0.5
    prediction = probability > threshold; rows = []
    for batch_index in range(prediction.shape[0]):
        record = {}
        for bone_index, bone in enumerate(BONES):
            pred_mask = prediction[batch_index, bone_index]; target_mask = truth[batch_index, bone_index]
            overlap = overlap_metrics(pred_mask, target_mask)
            record.update({f"dice_{bone}": overlap["dice"], f"iou_{bone}": overlap["iou"], f"hd95_mm_{bone}": hd95_mm(pred_mask, target_mask, spacing_xyz), f"assd_mm_{bone}": assd_mm(pred_mask, target_mask, spacing_xyz), f"empty_prediction_{bone}": overlap["empty_prediction"], f"invalid_target_{bone}": overlap["invalid_target"]})
        valid_dice = [record[f"dice_{bone}"] for bone in BONES if np.isfinite(record[f"dice_{bone}"])]
        valid_iou = [record[f"iou_{bone}"] for bone in BONES if np.isfinite(record[f"iou_{bone}"])]
        record["dice_macro"] = float(np.mean(valid_dice)) if valid_dice else float("nan"); record["iou_macro"] = float(np.mean(valid_iou)) if valid_iou else float("nan")
        rows.append(record)
    return rows


def two_largest_components(mask):
    labels, count = label(np.asarray(mask, dtype=bool))
    if count < 2: return None
    sizes = [(component, int((labels == component).sum())) for component in range(1, count + 1)]
    selected = sorted(sizes, key=lambda item: item[1], reverse=True)[:2]
    return labels == selected[0][0], labels == selected[1][0]


def minimum_component_gap_mm(mask, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    pair = two_largest_components(mask)
    if pair is None: return float("nan")
    first, second = pair
    return float(distance_transform_edt(~second, sampling=spacing_xyz)[first].min())


def component_bridge_metrics(prediction, target, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    _, pred_components = label(np.asarray(prediction, dtype=bool)); _, target_components = label(np.asarray(target, dtype=bool))
    return {"prediction_components": int(pred_components), "target_components": int(target_components), "component_agreement": bool(pred_components == target_components), "false_bridge": bool(target_components >= 2 and pred_components < target_components), "prediction_min_gap_mm": minimum_component_gap_mm(prediction, spacing_xyz), "target_min_gap_mm": minimum_component_gap_mm(target, spacing_xyz)}


def aggregate_subject_level(frame, metric_columns, subject_column="subject_id"):
    """Average knees within subject first, then average subjects so bilateral knees do not receive extra weight."""
    subject = frame.groupby(subject_column, as_index=False)[metric_columns].mean(numeric_only=True)
    return subject, subject[metric_columns].mean(numeric_only=True).to_dict()

In [ ]:
def find_project_root(start):
    for candidate in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").exists(): return candidate
    raise FileNotFoundError("project root not found")


ROOT = find_project_root(Path.cwd())
FOLD_ROOT = ROOT / "models" / STAGE2_SCHEMA / "fold_0"
CACHE_PATH = FOLD_ROOT / "frozen_feature_cache.pt"
CACHE_MANIFEST_PATH = FOLD_ROOT / "frozen_feature_cache_manifest.json"
OUTPUT_ROOT = ROOT / "models" / "decoders" / STAGE2_SCHEMA / "fold_0"
PROTOCOL_AMENDMENT_PATH = ROOT / "reports/agent_runs/stage2/s2_3_decoder_2x2_factorial" / "decoder_cv_protocol_amendment_v1.md"
P2_AUDIT_PATH = FOLD_ROOT / "feature_audit" / "p2" / "feature_audit_summary.json"
FOLD_QA_PATH = FOLD_ROOT / "feature_audit" / "independent_qa_verdict.json"


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""): h.update(chunk)
    return h.hexdigest()



def require_fold0_entry_qa():
    for required in (PROTOCOL_AMENDMENT_PATH, P2_AUDIT_PATH, FOLD_QA_PATH):
        if not required.is_file():
            raise FileNotFoundError(required)
    audit = json.loads(P2_AUDIT_PATH.read_text(encoding="utf-8"))
    verdict = json.loads(FOLD_QA_PATH.read_text(encoding="utf-8"))
    amendment_sha = sha256_file(PROTOCOL_AMENDMENT_PATH)
    audit_sha = sha256_file(P2_AUDIT_PATH)
    if audit.get("fold") != 0 or audit.get("structural_status") != "PASS" or not audit.get("encoder_unchanged") or audit.get("test_paths_opened") is not False:
        raise RuntimeError("fold-0 structural P2 audit failed")
    if verdict.get("schema_version") != "stage2_decoder_entry_qa_v1" or verdict.get("fold") != 0 or verdict.get("status") != "PASS" or verdict.get("scope") != "decoder_comparison":
        raise RuntimeError("fold-0 independent decoder-entry verdict is invalid")
    if verdict.get("audit_sha256") != audit_sha or verdict.get("protocol_amendment_sha256") != amendment_sha:
        raise RuntimeError("fold-0 independent verdict is not bound to current evidence")
    if not set(audit.get("warnings", [])).issubset(set(verdict.get("accepted_warnings", []))):
        raise RuntimeError("fold-0 audit warnings were not explicitly accepted")
    return {"audit_sha256": audit_sha, "quality_status": audit.get("quality_status"), "warnings": audit.get("warnings", []), "independent_verdict_sha256": sha256_file(FOLD_QA_PATH), "protocol_amendment_sha256": amendment_sha}

def tensor_sha256(tensor):
    value = tensor.detach().cpu().contiguous(); h = hashlib.sha256(); h.update(str(value.dtype).encode()); h.update(np.asarray(value.shape, dtype=np.int64).tobytes()); h.update(value.numpy().tobytes()); return h.hexdigest()


def canonical_sha256(payload): return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def state_sha256(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous(); digest.update(name.encode()); digest.update(str(tensor.dtype).encode()); digest.update(np.asarray(tensor.shape, dtype=np.int64).tobytes()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def peak_host_memory_bytes():
    if resource is None: return None
    maximum_rss = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    return maximum_rss if platform.system() == "Darwin" else maximum_rss * 1024


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_feature_cache():
    if not CACHE_PATH.is_file() or not CACHE_MANIFEST_PATH.is_file(): raise FileNotFoundError("fold-0 frozen feature cache and manifest are required")
    manifest = json.loads(CACHE_MANIFEST_PATH.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != STAGE2_SCHEMA or manifest.get("fold") != 0: raise RuntimeError("feature-cache schema/fold mismatch")
    if sha256_file(CACHE_PATH) != manifest.get("cache_sha256"): raise RuntimeError("feature-cache file hash mismatch")
    payload = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
    if payload.get("manifest_sha256") != manifest.get("manifest_sha256") or payload.get("frontend_checkpoint_sha256") != manifest.get("frontend_checkpoint_sha256"): raise RuntimeError("feature-cache provenance mismatch")
    expected = {row["sample_id"]: row for row in manifest["records"]}; records = {}
    for record in payload["records"]:
        sample_id = record["sample_id"]
        if sample_id not in expected: raise RuntimeError(f"unexpected cached sample: {sample_id}")
        hashes = [tensor_sha256(feature) for feature in record["features"]]
        if hashes != expected[sample_id]["feature_sha256"] or hashes != record["feature_sha256"]: raise RuntimeError(f"feature hash mismatch: {sample_id}")
        if tensor_sha256(record["target"]) != expected[sample_id]["target_sha256"]: raise RuntimeError(f"target hash mismatch: {sample_id}")
        shapes = [list(feature.shape) for feature in record["features"]]
        required_shapes = [[1, 64, 64, 64, 64], [1, 128, 32, 32, 32], [1, 256, 16, 16, 16], [1, 512, 8, 8, 8]]
        if shapes != required_shapes or list(record["target"].shape) != [4, 256, 256, 256]: raise RuntimeError(f"cached tensor contract mismatch: {sample_id}")
        records[sample_id] = record
    if set(records) != {"VSD_016_Left", "Case14_PartRight"}: raise RuntimeError("cache must contain exactly the approved healthy and fractured cases")
    return records, manifest


class DecoderBlock(nn.Module):
    def __init__(self, input_channels, output_channels, activation, residual):
        super().__init__(); self.residual = residual
        self.proj = (nn.Conv3d(input_channels, output_channels, 1) if input_channels != output_channels else nn.Identity()) if residual else None
        self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1); self.norm1 = nn.GroupNorm(8, output_channels); self.act1 = make_activation(activation, output_channels)
        self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1); self.norm2 = nn.GroupNorm(8, output_channels); self.act2 = make_activation(activation, output_channels)
    def forward(self, x):
        residual = self.proj(x) if self.residual else None
        x = self.act1(self.norm1(self.conv1(x))); x = self.norm2(self.conv2(x))
        if self.residual: x = x + residual
        return self.act2(x)


def make_activation(kind, channels):
    if kind == "relu": return nn.ReLU(inplace=True)
    if kind == "prelu": return nn.PReLU(channels)
    raise ValueError(f"unknown activation {kind!r}")


def block_for(arm, input_channels, output_channels):
    if arm not in ARM_FACTORS: raise ValueError(f"unknown arm {arm}")
    factors = ARM_FACTORS[arm]
    return DecoderBlock(input_channels, output_channels, factors["activation"], factors["residual"])


class Decoder3D(nn.Module):
    def __init__(self, arm):
        super().__init__(); self.arm = arm; c0, c1, c2, c3 = FEATURE_CHANNELS
        self.up3 = nn.ConvTranspose3d(c3, c2, 2, 2); self.dec3 = block_for(arm, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, 2, 2); self.dec2 = block_for(arm, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, 2, 2); self.dec1 = block_for(arm, c0 + c0, c0)
        self.refine128 = block_for(arm, c0, 32); self.output128 = nn.Conv3d(32, len(BONES), 1)
    def _run(self, module, value):
        if USE_ACTIVATION_CHECKPOINTING and self.training and value.requires_grad: return checkpoint.checkpoint(module, value, use_reentrant=False)
        return module(value)
    def forward(self, features):
        l0, l1, l2, l3 = features
        value = self._run(self.dec3, torch.cat([self.up3(l3), l2], 1)); value = self._run(self.dec2, torch.cat([self.up2(value), l1], 1)); value = self._run(self.dec1, torch.cat([self.up1(value), l0], 1))
        r = DECODER_LOGIT_RESOLUTION; value = self._run(self.refine128, F.interpolate(value, size=(r, r, r), mode="trilinear", align_corners=False)); logits = self.output128(value)
        return F.interpolate(logits, size=(TARGET_SIZE, TARGET_SIZE, TARGET_SIZE), mode="trilinear", align_corners=False)


def architecture_contract():
    models = {arm: Decoder3D(arm) for arm in ARMS}
    for arm, model in models.items():
        if any(isinstance(module, (nn.BatchNorm3d, nn.InstanceNorm3d)) for module in model.modules()): raise RuntimeError(f"forbidden normalization in {arm}")
        if sum(isinstance(module, nn.GroupNorm) for module in model.modules()) != 8: raise RuntimeError(f"expected 8 GroupNorm layers in {arm}")
    return {arm: {"activation": ARM_FACTORS[arm]["activation"], "residual": ARM_FACTORS[arm]["residual"], "parameters": sum(p.numel() for p in model.parameters()), "groupnorm_layers": sum(isinstance(m, nn.GroupNorm) for m in model.modules()), "output_policy": "four_logits_128_then_trilinear_256_align_corners_false"} for arm, model in models.items()}


def dice_bce_loss(logits, target):
    logits = logits.float(); target = target.float(); bce = F.binary_cross_entropy_with_logits(logits, target); probability = torch.sigmoid(logits).flatten(2); flattened = target.flatten(2); intersection = (probability * flattened).sum(-1); dice = (2 * intersection + 1.0) / (probability.sum(-1) + flattened.sum(-1) + 1.0); return 0.5 * bce + 0.5 * (1 - dice.mean())


@torch.no_grad()
def hard_dice(logits, target):
    prediction = (torch.sigmoid(logits.float()) > 0.5).float().flatten(2); target = (target > 0.5).float().flatten(2); intersection = (prediction * target).sum(-1); return (2 * intersection + 1e-6) / (prediction.sum(-1) + target.sum(-1) + 1e-6)

In [ ]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)


def make_scaler(): return torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported())


def capture_rng_state(): return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(state["cuda"])


def save_resume_checkpoint(path, epoch, global_update, model, optimizer, scheduler, scaler, arm, subset, feature_cache_sha, config_sha, parameter_sha=None, pre_update_logits_sha=None):
    torch.save({"schema_version": STAGE2_SCHEMA, "protocol_version": PROTOCOL_VERSION, "fold": 0, "arm": arm, "subset": subset, "epoch": epoch, "global_update": global_update, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng_state": capture_rng_state(), "feature_cache_sha256": feature_cache_sha, "config_sha256": config_sha, "parameter_sha256": parameter_sha, "pre_update_logits_sha256": pre_update_logits_sha}, path)


def resume_preflight(path, arm, features, expected_config_sha):
    saved = torch.load(path, map_location=DEVICE, weights_only=False)
    if saved.get("fold") != 0 or saved.get("arm") != arm or saved.get("config_sha256") != expected_config_sha: raise RuntimeError("save/resume provenance mismatch")
    reloaded = Decoder3D(arm).to(DEVICE); reloaded.load_state_dict(saved["model"], strict=True)
    parameter_sha = state_sha256(reloaded.state_dict())
    reloaded.eval()
    with torch.no_grad(): logits_sha = tensor_sha256(reloaded(features).float().cpu())
    if parameter_sha != saved.get("parameter_sha256"): raise RuntimeError("save/resume parameter hash mismatch")
    if logits_sha != saved.get("pre_update_logits_sha256"): raise RuntimeError("save/resume pre-update-logit hash mismatch")
    optimizer = torch.optim.Adam(reloaded.parameters(), lr=LEARNING_RATE); optimizer.load_state_dict(saved["optimizer"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS); scheduler.load_state_dict(saved["scheduler"])
    scaler = make_scaler(); scaler.load_state_dict(saved["scaler"]); restore_rng_state(saved["rng_state"])
    return reloaded, optimizer, scheduler, scaler, parameter_sha, logits_sha

def run_overfit(arm, subset, sample_ids, records, cache_manifest, entry_qa):
    seed_everything(); output_dir = OUTPUT_ROOT / arm / subset; output_dir.mkdir(parents=True, exist_ok=True)
    selected = [records[sample_id] for sample_id in sample_ids]; feature_cache_sha = cache_manifest["cache_sha256"]
    config = {"schema_version": STAGE2_SCHEMA, "protocol_version": PROTOCOL_VERSION, "stage": "decoder_capacity_gate", "fold": 0, "seed": SEED, "arm": arm, "subset": subset, "sample_ids": list(sample_ids), "manifest_sha256": cache_manifest["manifest_sha256"], "upstream_feature_cache_sha256": feature_cache_sha, "upstream_frontend_checkpoint_sha256": cache_manifest["frontend_checkpoint_sha256"], "input_feature_sha256": {row["sample_id"]: row["feature_sha256"] for row in cache_manifest["records"] if row["sample_id"] in sample_ids}, "augmentation": {"enabled": False, "reason": "deterministic_cached_capacity_gate"}, "hyperparameters": {"target_size": TARGET_SIZE, "batch_size": 1, "optimizer": "Adam", "learning_rate": LEARNING_RATE, "max_epochs": MAX_EPOCHS, "evaluation_every": EVALUATE_EVERY, "minimum_dice_each_case_bone": MINIMUM_DICE_EACH_CASE_BONE, "required_consecutive_passes": REQUIRED_CONSECUTIVE_PASSES, "deep_supervision": False, "amp": USE_AMP, "activation_checkpointing": USE_ACTIVATION_CHECKPOINTING}, "software": {"python": platform.python_version(), "torch": torch.__version__}, "hardware": {"device": str(DEVICE), "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None}, "output_paths": {"output_dir": str(output_dir)}}
    config["entry_qa"] = entry_qa; config["logit_resolution"] = DECODER_LOGIT_RESOLUTION; config_sha = canonical_sha256(config); (output_dir / "config.json").write_text(json.dumps(config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    model = Decoder3D(arm).to(DEVICE); optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS); scaler = make_scaler()
    parameter_count = sum(p.numel() for p in model.parameters()); history, global_update, consecutive, resume_pass, resume_logits_sha = [], 0, 0, False, None
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    started = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train(); epoch_loss = 0.0
        for record in selected:
            features = [feature.to(DEVICE, non_blocking=True) for feature in record["features"]]; target = record["target"].unsqueeze(0).float().to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context(): logits = model(features); loss = dice_bce_loss(logits, target)
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite decoder loss: {arm}/{subset}")
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update(); global_update += 1; epoch_loss += loss.item()
            if global_update == 5:
                model.eval(); parameter_sha = state_sha256(model.state_dict())
                with torch.no_grad(): pre_update_logits_sha = tensor_sha256(model(features).float().cpu())
                resume_path = output_dir / "resume_update005.pth"; save_resume_checkpoint(resume_path, epoch, global_update, model, optimizer, scheduler, scaler, arm, subset, feature_cache_sha, config_sha, parameter_sha, pre_update_logits_sha)
                model, optimizer, scheduler, scaler, resumed_parameter_sha, resume_logits_sha = resume_preflight(resume_path, arm, features, config_sha)
                if resumed_parameter_sha != parameter_sha or resume_logits_sha != pre_update_logits_sha: raise RuntimeError("save/resume hash verification failed")
                resume_pass = True; model.train()
        scheduler.step()
        if epoch % EVALUATE_EVERY == 0 or epoch == MAX_EPOCHS - 1:
            model.eval(); case_dice = {}
            with torch.no_grad():
                for record in selected:
                    features = [feature.to(DEVICE) for feature in record["features"]]; target = record["target"].unsqueeze(0).float().to(DEVICE); case_dice[record["sample_id"]] = hard_dice(model(features), target)[0].cpu().numpy()
            minimum = np.stack(list(case_dice.values()), axis=0).min(axis=0); passed_now = bool((minimum >= MINIMUM_DICE_EACH_CASE_BONE).all()); consecutive = consecutive + 1 if passed_now else 0
            row = {"epoch": epoch, "mean_loss": epoch_loss / len(selected), **{f"minimum_dice_{bone}": float(minimum[i]) for i, bone in enumerate(BONES)}, "consecutive_passes": consecutive}
            for sample_id, values in case_dice.items():
                for index, bone in enumerate(BONES): row[f"dice_{sample_id}_{bone}"] = float(values[index])
            history.append(row); pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False); print(arm, subset, row)
            if consecutive >= REQUIRED_CONSECUTIVE_PASSES: break
    gate_pass = consecutive >= REQUIRED_CONSECUTIVE_PASSES
    if not resume_pass: raise RuntimeError("save/resume preflight did not execute")
    if not gate_pass: raise RuntimeError(f"overfit gate failed: {arm}/{subset}")
    peak = int(torch.cuda.max_memory_allocated()) if DEVICE.type == "cuda" else None; total = int(torch.cuda.get_device_properties(DEVICE).total_memory) if DEVICE.type == "cuda" else None; headroom = None if total is None else 1.0 - peak / total; peak_host = peak_host_memory_bytes()
    if headroom is not None and headroom < 0.10: raise RuntimeError(f"GPU headroom below 10%: {headroom:.3f}")
    final_path = output_dir / "decoder_gate_pass.pth"; save_resume_checkpoint(final_path, epoch, global_update, model, optimizer, scheduler, scaler, arm, subset, feature_cache_sha, config_sha, state_sha256(model.state_dict()), None)
    final = history[-1]; history_frame = pd.DataFrame(history); figure, axis = plt.subplots(figsize=(8, 5))
    for bone in BONES: axis.plot(history_frame.epoch, history_frame[f"minimum_dice_{bone}"], label=bone)
    axis.axhline(MINIMUM_DICE_EACH_CASE_BONE, color="black", linestyle="--", label="gate"); axis.set(xlabel="epoch", ylabel="minimum hard Dice", ylim=(0, 1), title=f"{arm} - {subset}"); axis.grid(alpha=0.25); axis.legend(); figure.tight_layout(); qa_path = output_dir / "capacity_curve.png"; figure.savefig(qa_path, dpi=160); plt.close(figure)
    return {"arm": arm, "subset": subset, "sample_ids": sample_ids, "pass": gate_pass, "resume_pass": resume_pass, "resume_parameter_sha256": resumed_parameter_sha, "resume_logits_sha256": resume_logits_sha, "config_sha256": config_sha, "feature_cache_sha256": feature_cache_sha, "parameters": parameter_count, "epochs_completed": epoch + 1, "global_updates": global_update, "peak_gpu_bytes": peak, "peak_host_bytes": peak_host, "total_gpu_bytes": total, "gpu_headroom_fraction": headroom, "wall_seconds": round(time.time() - started, 1), "qa_figure": str(qa_path), "checkpoint_sha256": sha256_file(final_path), **{f"minimum_dice_{bone}": final[f"minimum_dice_{bone}"] for bone in BONES}}


def run_foundation_gate():
    if DEVICE.type != "cuda": raise RuntimeError("the full 256^3 decoder gate requires the HPC GPU")
    entry_qa = require_fold0_entry_qa(); records, cache_manifest = load_feature_cache(); architecture = architecture_contract(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True); results = []
    for subset, sample_ids in SUBSETS.items():
        for arm in ARMS: results.append(run_overfit(arm, subset, sample_ids, records, cache_manifest, entry_qa))
    frame = pd.DataFrame([{key: value for key, value in row.items() if key != "sample_ids"} for row in results]); frame.to_csv(OUTPUT_ROOT / "foundation_summary.csv", index=False)
    summary = {"schema_version": STAGE2_SCHEMA, "protocol_version": PROTOCOL_VERSION, "fold": 0, "architecture": architecture, "entry_qa": entry_qa, "feature_cache_sha256": cache_manifest["cache_sha256"], "expected_results": 8, "results": results, "pass": len(results) == 8 and all(row["pass"] and row["resume_pass"] for row in results)}
    (OUTPUT_ROOT / "foundation_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"); return summary

In [ ]:
print("architecture contract:", architecture_contract())
if RUN_DECODER_GATE:
    display(run_foundation_gate())
else:
    print("Definitions loaded. Obtain fold-0 independent entry QA, then set RUN_DECODER_GATE=True on HPC.")
